# Lab 2 Assignment: Independent Tool Use Review

Use this notebook after you finish `03b_lab_notebook.ipynb`. The goal here is to apply the same three-tool workflow with less guidance so you can show that you understand how to choose tool calls, inspect outputs, and defend an evidence-based conclusion.

In this assignment, you will:
- locate candidate media with the same local forensic tools
- choose at least two image files to inspect
- build your own evidence bundle and manual report
- compare your manual workflow with `ToolAgent`
- explain one `ToolAgent` choice you would accept or correct

Keep `03b_lab_notebook.ipynb` nearby if you want to review the guided examples, but complete the choices in this notebook yourself.

In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and loads the case data.
LAB_NAME = 'lab2_tool_use_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

load_dotenv(env_path, override=True)

MODEL = os.getenv("MODEL")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f"MODEL or OLLAMA_BASE_URL is missing from {env_path}")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

data_dir = lab_dir / "data"
if not (data_dir / "artifact_manifest.json").exists():
    raise FileNotFoundError("Could not find lab2_tool_use_pattern/data")

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

def load_csv_records(path: Path) -> list[dict]:
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))

artifact_manifest = json.loads((data_dir / "artifact_manifest.json").read_text())
media_index = load_csv_records(data_dir / "media_index.csv")
image_metadata = load_csv_records(data_dir / "image_metadata.csv")
vehicle_detections = load_csv_records(data_dir / "vehicle_detections.csv")
listing_drafts = json.loads((data_dir / "listing_drafts.json").read_text())
chain_of_custody = load_csv_records(data_dir / "chain_of_custody.csv")

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)


## Task 1

Review the assignment question and the required report format. This assignment uses the same five-part structure as the guided notebook, but here you must choose the image evidence yourself.

Important assignment rule: inspect **at least two** image files returned by `list_media_files` before you finalize your conclusion.

In [ ]:
ASSIGNMENT_QUESTION = (
    "Which image provides the strongest support that the stolen black SUV was being prepared "
    "for an online sale after January 2, 2026, and does the overall evidence support a "
    "confirmed, likely, or unconfirmed conclusion?"
)

REPORT_FORMAT = """
Return:
(1) tool-call log
(2) strongest timestamp evidence
(3) strongest vehicle-match evidence
(4) conclusion label (confirmed, likely, or unconfirmed) with confidence 0-1 per major claim
(5) explicit evidence mapping and limits
""".strip()

assignment_summary = {
    "assignment_question": ASSIGNMENT_QUESTION,
    "required_report_format": REPORT_FORMAT,
    "minimum_image_reviews": 2,
}

assignment_summary

## Task 2

Review the staged case manifest again before you start calling tools. This keeps the incident window and artifact packet visible while you make your own decisions.

In [ ]:
artifact_manifest

## Task 3

Load the same three forensic tools from the guided notebook. The tool set is unchanged; the main difference in this assignment is that you must decide how to use it.

In [ ]:
from agentic_patterns.tool_pattern.tool import tool
from agentic_patterns.tool_pattern.tool_agent import ToolAgent

media_by_name = {row["file_name"]: row for row in media_index}
metadata_by_name = {row["file_name"]: row for row in image_metadata}
detections_by_name = {row["file_name"]: row for row in vehicle_detections}


def normalize_root(root: str) -> str:
    normalized = root.strip().strip("/")
    if normalized.lower() == "gallery":
        return "DCIM/Camera"
    return normalized


def as_json(payload) -> str:
    return json.dumps(payload, indent=2)


@tool
def list_media_files(root: str, date_from: str):
    """
    List media files from the requested folder on or after a given UTC timestamp.

    Args:
        root (str): Folder name such as "DCIM/Camera" or "gallery/".
        date_from (str): Lower UTC bound in ISO-8601 format.
    """
    normalized_root = normalize_root(root)
    matches = [
        {
            "file_name": row["file_name"],
            "folder": row["folder"],
            "created_at_utc": row["created_at_utc"],
            "path": row["path"],
            "size_bytes": row["size_bytes"],
        }
        for row in media_index
        if row["created_at_utc"] >= date_from and row["folder"].startswith(normalized_root)
    ]
    return as_json(matches)


@tool
def inspect_image_evidence(file_name: str, case_description: str):
    """
    Inspect one candidate image by combining file details, image metadata, vehicle detections,
    and a comparison to the case vehicle description.

    Args:
        file_name (str): Image filename such as "IMG_2044.jpg".
        case_description (str): Known case description such as "black SUV with roof rack".
    """
    media_record = media_by_name.get(file_name)
    metadata_record = metadata_by_name.get(file_name)
    detection_record = detections_by_name.get(file_name)

    if media_record is None:
        return as_json({"error": f"No media record found for {file_name}"})
    if metadata_record is None:
        return as_json({"error": f"No metadata found for {file_name}"})
    if detection_record is None:
        return as_json({"error": f"No detection found for {file_name}"})

    description = case_description.lower()
    matching = []
    conflicting = []

    if "black" in description:
        if detection_record["color"].lower() == "black":
            matching.append("black")
        else:
            conflicting.append(f"color={detection_record['color']}")
    if "suv" in description:
        if detection_record["body_type"].lower() == "suv":
            matching.append("SUV")
        else:
            conflicting.append(f"body_type={detection_record['body_type']}")
    if "roof rack" in description:
        if detection_record["roof_rack"].lower() == "yes":
            matching.append("roof rack")
        else:
            conflicting.append("roof rack not detected")

    if len(matching) == 3 and not conflicting:
        match_strength = "strong"
    elif len(matching) >= 2:
        match_strength = "moderate"
    else:
        match_strength = "weak"

    return as_json(
        {
            "file_name": file_name,
            "file_record": {
                "path": media_record["path"],
                "folder": media_record["folder"],
                "created_at_utc": media_record["created_at_utc"],
                "size_bytes": media_record["size_bytes"],
            },
            "image_metadata": metadata_record,
            "vehicle_detection": detection_record,
            "feature_comparison": {
                "case_description": case_description,
                "matching_features": matching,
                "conflicting_features": conflicting,
                "match_strength": match_strength,
            },
        }
    )


@tool
def inspect_listing_records(date_from: str):
    """
    Inspect saved listing-draft records created on or after a given UTC timestamp.

    Args:
        date_from (str): Lower UTC bound in ISO-8601 format.
    """
    matches = [draft for draft in listing_drafts if draft["created_at_utc"] >= date_from]
    return as_json(matches)


forensic_tools = [
    list_media_files,
    inspect_image_evidence,
    inspect_listing_records,
]


## Task 4

Inspect the tool schemas if you need a reminder of the expected arguments. If you want concrete examples too, reopen `03b_lab_notebook.ipynb` and review its quick tool examples.

In [ ]:
tool_catalog = {tool.name: json.loads(tool.fn_signature) for tool in forensic_tools}
tool_catalog

## Task 5

Start with `list_media_files` so you can see which images are available during the incident window. You will use this output to choose the files you want to inspect next.

In [ ]:
date_from = artifact_manifest["incident_window_utc"]["start"]
candidate_media = json.loads(list_media_files.run(root="DCIM/Camera", date_from=date_from))

candidate_media

## Task 6

Replace the two placeholder file names below with image files returned by `candidate_media`. Then run the cell to inspect both images, retrieve the listing records, and build your evidence bundle.

You may inspect more than two files if you want, but the assignment requires at least two.

In [ ]:
student_case_description = "black SUV with roof rack"
student_image_choices = [
    "REPLACE_WITH_FIRST_FILE",
    "REPLACE_WITH_SECOND_FILE",
]

placeholder_prefix = "REPLACE_WITH_"
if any(name.startswith(placeholder_prefix) for name in student_image_choices):
    raise ValueError("Replace both placeholder file names with values returned by candidate_media.")

student_image_choices = list(dict.fromkeys(student_image_choices))
if len(student_image_choices) < 2:
    raise ValueError("Choose at least two candidate image files for this assignment.")

valid_file_names = {row["file_name"] for row in candidate_media}
unknown_files = [name for name in student_image_choices if name not in valid_file_names]
if unknown_files:
    raise ValueError(f"These file names were not returned by list_media_files: {unknown_files}")

student_image_evidence = {
    file_name: json.loads(
        inspect_image_evidence.run(file_name=file_name, case_description=student_case_description)
    )
    for file_name in student_image_choices
}
listing_results = json.loads(inspect_listing_records.run(date_from=date_from))

# Keep a record of the manual sequence you chose so it can go into the final report.
student_tool_log = [
    {"tool": "list_media_files", "arguments": {"root": "DCIM/Camera", "date_from": date_from}},
    *[
        {"tool": "inspect_image_evidence", "arguments": {"file_name": file_name, "case_description": student_case_description}}
        for file_name in student_image_choices
    ],
    {"tool": "inspect_listing_records", "arguments": {"date_from": date_from}},
]

student_evidence_bundle = {
    "candidate_media": candidate_media,
    "image_evidence": student_image_evidence,
    "listing_results": listing_results,
}

student_evidence_bundle

## Task 7

Build the manual report prompt from the evidence you collected. This is the same pattern you used in the guided notebook, but now the tool-call log and evidence bundle come from your own choices.

In [ ]:
manual_prompt = f"""
You are a careful forensic analyst. Stay inside the evidence.
Use the manually collected tool outputs below to answer the assignment question.

Assignment question:
{ASSIGNMENT_QUESTION}

Tool-call log:
{json.dumps(student_tool_log, indent=2)}

Tool outputs:
{json.dumps(student_evidence_bundle, indent=2)}

{REPORT_FORMAT}
""".strip()

print(manual_prompt[:1800])

## Task 8

Ask the model to write the manual report. The `system` role sets the overall behavior, and the `user` role carries the specific prompt built from your evidence bundle.

In [ ]:
# The system message sets the overall behavior. The user message carries the specific assignment prompt.
manual_report = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a careful forensic analyst. Distinguish observed evidence from conclusions and state unresolved limits explicitly.",
        },
        {
            "role": "user",
            "content": manual_prompt,
        },
    ],
).choices[0].message.content

print(manual_report)

## Task 9

Now let `ToolAgent` investigate the same assignment question. Compare the tool choices it makes with the sequence you chose manually.

For this assignment, the agent should still inspect at least two candidate image files before it finalizes its conclusion.

In [ ]:
tool_agent = ToolAgent(tools=forensic_tools, client=client, model=MODEL)

agent_user_msg = f"""
Use the available forensic tools to answer this assignment question.

Assignment question:
{ASSIGNMENT_QUESTION}

Required core sequence:
list_media_files -> inspect_image_evidence -> inspect_listing_records

Assignment rules:
- inspect at least two candidate image filenames returned by list_media_files before concluding
- use only file_name values returned by list_media_files when you call inspect_image_evidence
- the media folder is DCIM/Camera
- inspect_listing_records already reads the saved listing drafts for this lab
- the known vehicle description is: black SUV with roof rack
- focus on the strongest image candidate, but mention competing evidence if it weakens the conclusion

{REPORT_FORMAT}
""".strip()

agent_report = tool_agent.run(user_msg=agent_user_msg)
print(agent_report)

## Task 10: Reflection and Submission

After you read both reports, replace the empty strings in the next cell. Use short plain-language answers.

Focus especially on one `ToolAgent` choice you would accept or correct based on the tool schema, argument validity, or missing evidence.

In [ ]:
assignment_reflection = {
    "strongest_image_you_selected": "",
    "manual_conclusion_label": "",
    "toolagent_conclusion_label": "",
    "one_toolagent_choice_to_accept_or_correct": "",
    "why_that_choice_should_be_accepted_or_corrected": "",
    "which_report_you_trust_more_and_why": "",
}

required_text_fields = list(assignment_reflection.keys())
missing = [key for key in required_text_fields if not str(assignment_reflection[key]).strip()]

print(json.dumps(assignment_reflection, indent=2))

if missing:
    print(f"\nComplete these fields before submitting: {missing}")

## Submission

Save the notebook with:
- the `candidate_media` output from Task 5
- the completed `student_evidence_bundle` from Task 6
- the manual report from Task 8
- the `ToolAgent` report from Task 9
- the completed `assignment_reflection` from Task 10